In [16]:
from pathlib import Path
import sys
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
repo_root

WindowsPath('C:/Users/leungk/OneDrive - EllisDon Corporation/Documents/Other/github_repos/CBG_analysis')

# Meal Aggregation and Model Table
Aggregate item-level macros to meals, add time-based features, and prep tables for relating meals to post-meal glucose outcomes.

In [17]:
from pathlib import Path
import pandas as pd
from src import io_excel, parse_foods, defaults, aggregate, usda_client

repo_root = Path('..').resolve()
excel_path = repo_root / 'data' / 'source_data' / '20251218_Trudy_Meals.xlsx'
defaults_path = repo_root / 'config' / 'defaults_food_items.yaml'
api_keys_path = repo_root / 'config' / 'api_keys.json'
cache_path = repo_root / 'notebook' / 'cache_data' / 'parquet' / 'food_nutrition_cache.parquet'

print(f'Loading clean events from {excel_path}')
clean_events = io_excel.load_clean_events(excel_path)
print(f'Clean events: {len(clean_events)} rows')

Loading clean events from C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\source_data\20251218_Trudy_Meals.xlsx
Clean events: 206 rows


C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\src\io_excel.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_parsed = pd.to_datetime(time_series, errors="coerce").dt.time


In [18]:
# Parse foods and apply defaults
items = parse_foods.explode_food_items(clean_events)
print(f'Exploded to items: {len(items)} rows')

cfg = defaults.load_defaults_config(defaults_path)
items = defaults.apply_default_rules(items, cfg)
assumed_rate = items['assumed_100g_flag'].mean() if 'assumed_100g_flag' in items else 0.0
print(f'Defaults applied; assumed_100g_flag rate: {assumed_rate:.3f}')

Exploded to items: 644 rows
Defaults applied; assumed_100g_flag rate: 0.000


In [19]:
# Convert to grams and inspect
items = aggregate.compute_grams(items)
print('Converted to grams; grams_final stats:')
print(items['grams_final'].describe())

Converted to grams; grams_final stats:
count    644.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: grams_final, dtype: float64


In [20]:
# USDA enrich and cache
items, cache = usda_client.enrich_items_with_usda(items, api_keys_path, cache_path)
print('usda_match_status counts:')
print(items['usda_match_status'].value_counts(dropna=False))

usda_match_status counts:
usda_match_status
cache_or_api    491
missing         153
Name: count, dtype: int64


C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\src\usda_client.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  "usda_description": result.get("description"),


In [21]:
# Compute item macros
items = aggregate.compute_item_macros(items)
macro_cols = [c for c in ['meal_id', 'food_name_std', 'grams_final', 'protein_g', 'carb_g', 'fat_g', 'calories'] if c in items]
print('Item macros preview:')
print(items[macro_cols].head(10))

Item macros preview:
   meal_id               food_name_std  grams_final  protein_g    fat_g  \
0        1                      2 eggs        100.0       4.13    2.830   
1        1              1 5 oz of beef        100.0      21.40    5.000   
2        1             2 cups choy sum        100.0       1.02    0.234   
3        2                         NaN        100.0        NaN      NaN   
4        3                1 cup quinoa        100.0       4.40    1.920   
5        3                      1 kiwi        100.0       0.00    0.000   
6        3     avocado oil for cooking        100.0       0.00  100.000   
7        3         2 cups chicken soup        100.0       2.53    0.550   
8        3         1 5 piece pork chop        100.0      22.80    5.480   
9        3  1 cup stir fry green beans        100.0       0.00  100.000   

   calories  
0     112.0  
1     137.0  
2       NaN  
3       NaN  
4     120.0  
5      47.0  
6     884.0  
7      37.0  
8       NaN  
9     884.0  

In [22]:
# Build meal features
meal_features = aggregate.build_meal_features(items, clean_events)
print(f'Meal features rows: {len(meal_features)}')
print(meal_features.head(10))

Meal features rows: 205
   meal_id  calories  carbs_g  protein_g    fat_g  items_count  \
0        1     249.0   21.110      26.55    8.064            3   
1        2       0.0    0.000       0.00    0.000            1   
2        3    2562.0   57.448      51.13  260.450            7   
3        4       0.0    0.000       0.00    0.000            1   
4        5    2358.0   18.700      21.40  252.500            6   
5        6    1183.0   75.470      78.61   81.050            6   
6        7     474.0   87.010      19.10    7.254            4   
7        8       0.0    0.000       0.00    0.000            1   
8        9     112.0   17.600       4.13    2.830            3   
9       10    2018.0  130.270      75.12  143.700            7   

   assumed_items_count  missing_usda_items_count            datetime  \
0                    3                         1 2025-01-12 00:00:00   
1                    0                         1 2025-01-15 00:00:00   
2                    6           

In [23]:
# Build model table with lags
model_table = aggregate.build_model_table(meal_features)
print(f'Model table rows: {len(model_table)}')
print(model_table.head(10))

Model table rows: 205
   meal_id  calories  carbs_g  protein_g    fat_g  items_count  \
0        1     249.0   21.110      26.55    8.064            3   
1        2       0.0    0.000       0.00    0.000            1   
2        3    2562.0   57.448      51.13  260.450            7   
3        4       0.0    0.000       0.00    0.000            1   
4        5    2358.0   18.700      21.40  252.500            6   
5        6    1183.0   75.470      78.61   81.050            6   
6        7     474.0   87.010      19.10    7.254            4   
7        8       0.0    0.000       0.00    0.000            1   
8        9     112.0   17.600       4.13    2.830            3   
9       10    2018.0  130.270      75.12  143.700            7   

   assumed_items_count  missing_usda_items_count            datetime  \
0                    3                         1 2025-01-12 00:00:00   
1                    0                         1 2025-01-15 00:00:00   
2                    6             

In [24]:
# Model table preview
print('Selected model_table columns preview:')
print(model_table[['meal_id', 'cbg_post', 'cbg_prev_same_day', 'hour_sin', 'hour_cos']].head(20))

Selected model_table columns preview:
    meal_id  cbg_post  cbg_prev_same_day  hour_sin      hour_cos
0         1       6.5                NaN  0.000000  1.000000e+00
1         2       6.0                NaN  0.000000  1.000000e+00
2         3       NaN                NaN -1.000000 -1.836970e-16
3         4       NaN                NaN  0.000000  1.000000e+00
4         5       NaN                NaN  0.500000 -8.660254e-01
5         6       NaN                NaN  0.500000 -8.660254e-01
6         7       NaN                NaN -0.965926 -2.588190e-01
7         8       NaN                NaN  0.000000  1.000000e+00
8         9       NaN                NaN  0.500000 -8.660254e-01
9        10       NaN                NaN -0.707107 -7.071068e-01
10       11       NaN                NaN -1.000000 -1.836970e-16
11       12       NaN                NaN  0.000000  1.000000e+00
12       13       NaN                NaN  0.000000  1.000000e+00
13       14       NaN                NaN -0.500000 -